In [20]:
import os
import re
import time
from typing import TypedDict

from langchain_community.document_loaders import YoutubeLoader
from langchain_mistralai import ChatMistralAI,MistralAIEmbeddings
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph,START,END
from langchain_core.tools import tool
from deep_translator import GoogleTranslator

In [21]:
load_dotenv()
llm=ChatMistralAI(model_name='mistral-large-2512')
llm_embeddings = MistralAIEmbeddings(model='mistral-embed')

In [22]:
url = 'https://youtu.be/o6rDvqXtAGE?si=dBZaByCdqWCyG0Dp'
#url = "https://www.youtube.com/watch?v=KyCtfQjkuR4&list=PLQxDHpeGU14Blorx3Ps1eZJ4XvKET1_vx&index=2"

In [23]:
urls = YoutubeLoader.from_youtube_url(url,language='hi'or'en')
content = urls.load()[0].page_content
#content

In [24]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap = 200)
text_split = text_splitter.split_text(content)
type(text_split)

list

In [25]:
len(text_split)

6

In [26]:
text_split

['सिंह मुझे तुमसे कुछ बात करनी है मेरी तरफ देखो गौर से देखो तुम्हें क्या दिखाई देता है ली चढ़ता मैं व्यक्तित्व की बात नहीं कर रहा हूं ने मान भी ली नहीं हम मतलब है कि आंखन के नीचे थोड़ी झुरिया आ गई है और थोड़ी सूजन आ गई की खाल लटक के नीचे आ रही मैं त्वचा की बात नहीं कर रहा हूं टचे आदमी मनोहर यस सर तुम बताओ गौर से देखो तुम्हें क्या दिखाई देता है सर मुझे आपकी आंखों में नमी दुनिया भर का दुख और और परेशानियां दिख रही है सर क्या बात है मनोहर क्या बात है तुम तुम तुम तुम पास हो गए एक सेकंड झुन के बाद हम यही टॉपिक पे आ रहे धीरे धीरे नमी दुख दर् सब बताती हमें मौका नहीं मिलू मनोहर तुम्हें पता कैसे चला हर शादीशुदा आदमी का चेहरा ऐसे दिखता है सर नहीं नहीं नहीं नहीं तुम फेल हो गए यही से हमने ऐसी बात नहीं करी कि फेल हो जाओ बाद में क पाएगा हमें आप खोल के बताओ आपकी प्रॉब्लम क्या चल रही हम सब सॉल्व करेंगे तुमसे क्या छुपाना पुशिंग मेरे पास कुछ ही साल बचे हैं सबट साइड इफेक्ट इतने जल्दी आप टपकने वाली पार्टी तो हो नहीं रिटायरमेंट को ज्यादा साल नहीं बचे हैं बेवकूफ ऑलमोस्ट फिनिशिंग लाइन पर है लेकिन जाते जाते मै

In [29]:
def fatch_transcript(url):
    urls = YoutubeLoader.from_youtube_url(url,language='hi'or'en')
    content = urls.load()[0].page_content
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap = 200)
    text_split = text_splitter.split_text(content)
    prompt=ChatPromptTemplate.from_template("""
        You are an expert translator with deep cultural and linguistic knowledge.
        I will provide you with a transcript. Your task is to translate it into English with absolute accuracy, preserving:
        - Full meaning and context (no omissions, no additions).
        - Tone and style (formal/informal, emotional/neutral as in original).
        - Nuances, idioms, and cultural expressions (adapt appropriately while keeping intent).
        - Speaker’s voice (same perspective, no rewriting into third-person).
        Do not summarize or simplify. The translation should read naturally in the target language but stay as close as possible to the original intent.

        Transcript:{transcript}
        """)
    #Runnable chain
    chain = prompt|llm
    #Run chain
    response = chain.invoke(text_split)
    return response.content
f_transcript = fatch_transcript(url)

In [30]:
type(f_transcript)

str

In [46]:
text = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap = 200)
text_sp = text.split_text(f_transcript)
type(text_sp)

list

In [47]:
len(text_sp)

9

In [49]:
embeddings = MistralAIEmbeddings(model='mistral-embed')
vactor_store = Chroma.from_texts(text_sp, embeddings)


In [50]:
vactor_store

In [53]:
def create_vector_store(text):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap = 200
        )
    text_split = text_splitter.split_text(text)

    embedding= MistralAIEmbeddings(model="mistral-embed")

    vector_store = Chroma.from_texts(texts=text_split,embedding=embedding)
    return vector_store

f = create_vector_store(f_transcript)

In [57]:
question = "what is the 2 main topic of this video"

In [61]:
results= f.similarity_search(question,k=4)
context_text = (results)
context_text

[Document(id='c2a11b01-60db-46d9-b419-7ec712163688', metadata={}, page_content='सिंह मुझे तुमसे कुछ बात करनी है मेरी तरफ देखो गौर से देखो तुम्हें क्या दिखाई देता है ली चढ़ता मैं व्यक्तित्व की बात नहीं कर रहा हूं ने मान भी ली नहीं हम मतलब है कि आंखन के नीचे थोड़ी झुरिया आ गई है और थोड़ी सूजन आ गई की खाल लटक के नीचे आ रही मैं त्वचा की बात नहीं कर रहा हूं टचे आदमी मनोहर यस सर तुम बताओ गौर से देखो तुम्हें क्या दिखाई देता है सर मुझे आपकी आंखों में नमी दुनिया भर का दुख और और परेशानियां दिख रही है सर क्या बात है मनोहर क्या बात है तुम तुम तुम तुम पास हो गए एक सेकंड झुन के बाद हम यही टॉपिक पे आ रहे धीरे धीरे नमी दुख दर् सब बताती हमें मौका नहीं मिलू मनोहर तुम्हें पता कैसे चला हर शादीशुदा आदमी का चेहरा ऐसे दिखता है सर नहीं नहीं नहीं नहीं तुम फेल हो गए यही से हमने ऐसी बात नहीं करी कि फेल हो जाओ बाद में क पाएगा हमें आप खोल के बताओ आपकी प्रॉब्लम क्या चल रही हम सब सॉल्व करेंगे तुमसे क्या छुपाना पुशिंग मेरे पास कुछ ही साल बचे हैं सबट साइड इफेक्ट इतने जल्दी आप टपकने वाली पार्टी तो हो नहीं रिटायरमेंट को

In [ ]:
retriever = vactor_store.as_retriever(search_type='similarity',search_kwargs={'k':4})

In [ ]:
retriever.invoke('policy')[0].page_content

'सारी जिम्मेदारी है भाई अगर भगवान ना करे वैसे होन नहीं है हमें अपने हाथन पर भरोस अगर जो हम नहीं दब पाए तो सारी की सारी जिम्मेदारी हम ले रहे सजा भी हम काटेंगे आप आच ना आने देंगे एक्सलेंट आइडिया च भी आपकी पट भी आपका न बैड आइडिया पु न बैड आइडिया री गुड रीड री गुड थैंक यू थैंक य डन न डन डन ना डन न क्या रे मेरे बारे में चल रही है [संगीत] [संगीत] क्या ज में तो पासपोर्ट जरूर पूरी कागजात नई शख्सियत और कछु पैसा है समझ र हो पंख फैला और उड़ [संगीत] जा इस मेहरबानी की वजह बस कर दे छोटी बात अरे अब तू आजाद परिंदो है जा दूसरी दुनिया में उड़ जा दूसरी कंट्री फटाफट कबूतर चा चा चा कबूतर चा चा चाचा ए कमिश्नर क्या तुम्हें भी कोई दिक्कत नहीं है नॉट एट ऑल नॉट एट ऑल अब तुम आजाद हो लेकिन आपने तो मुझे पकड़ने के लिए इनाम रखा था फिर आप ही मुझे यहां से भगा रहे हो क्यों आवाज नीचे यह फैसला हप्पू सिंह का था और मैं हप्पू सिंह के फैसले की कदर करता हूं वैसे हपू सर का दिल मोम का है वोह किसी को कैद में नहीं देख सकते मुझे पता नहीं था कि पुलिस वाले का दिल भी इतना नाजुक होता है व दादा अे तो पता चल निकल ले [संगीत] जोरावर की'